# C9. Speed

Is the world fast enough to explore? The final world against its distilled students.

See `docs/plans/experiments-plan.md`, entry C9. Results: `artifacts/results/world_speed/`.

## Run

From the repo root. `uv run erosion-sim list` shows what is already trained and evaluated.

**1. Data.** Skip any that already exist.

Local:

```sh
(cd datagen && uv run erosion-datagen generate erosion_master/main_512)
```

Cluster:

```sh
sbatch slurm/datagen.sbatch erosion_master/main_512
```

**2. Train**, in this order. Each resumes if stopped; a finished model is skipped. `coarse_student` needs `final_coarse_seed0` finished first. `fine_student` needs `final_fine_seed0` finished first.

Local:

```sh
uv run erosion-sim train final_coarse_seed0
uv run erosion-sim train final_fine_seed0
uv run erosion-sim train coarse_student
uv run erosion-sim train fine_student
```

Cluster:

```sh
sbatch slurm/train.sbatch final_coarse_seed0
sbatch slurm/train.sbatch final_fine_seed0
sbatch slurm/train.sbatch coarse_student
sbatch slurm/train.sbatch fine_student
```

**3. Evaluate**, once training has finished. It refuses to replace existing results.

Local:

```sh
uv run erosion-sim evaluate world_speed final_seed0
uv run erosion-sim evaluate world_speed students_1_step
uv run erosion-sim evaluate world_speed students_2_steps
uv run erosion-sim evaluate world_speed no_change
```

Cluster:

```sh
sbatch slurm/evaluate.sbatch world_speed final_seed0
sbatch slurm/evaluate.sbatch world_speed students_1_step
sbatch slurm/evaluate.sbatch world_speed students_2_steps
sbatch slurm/evaluate.sbatch world_speed no_change
```

Then run the cells below.

How to read the tables and plots: [`docs/guides/world.md`](../../docs/guides/world.md).

In [ ]:
import json

from erosion_sim.erosion import plots
from erosion_sim.evaluation.results import load
from erosion_sim.paths import model_dir
from erosion_sim.training.train import plot_history

EVALUATION = "world_speed"  # configs/evaluations/world_speed.yaml
CASCADES = ["final_seed0", "students_1_step", "students_2_steps"]  # configs/predictors/
BASELINES = ["no_change"]  # configs/predictors/
PREDICTORS = CASCADES + BASELINES
MODELS = ["final_coarse_seed0", "final_fine_seed0", "coarse_student", "fine_student"]  # every trained model these use, for the training plots

results = load(EVALUATION, *PREDICTORS)
{p: {k: m[k] for k in ("time", "code", "slurm_job")} for p, m in results.manifest["predictors"].items()}

## Training

In [ ]:
for name in MODELS:
    print(name, model_dir(name))
    plot_history(json.loads((model_dir(name) / "history.json").read_text()))

## Cascades, levels alone, and baselines, on the runs' interiors

In [ ]:
plots.table(results, plots.TABLE + ('chunk_edge_rmse', 'seconds_per_query'))

In [ ]:
if "dT" in results.scalars["by"]:
    plots.by_table(results, "dT", "skill", kinds=("cascade", "baseline"))

## Seams

How much neighboring windows disagree before blending, per level and phase, as a share of the change.

In [ ]:
for world, levels in results.scalars["seams"].items():
    for level, phases in levels.items():
        print(world, level, [f"{p['relative']:.1%}" for p in phases])

## Examples

In [ ]:
plots.plot_examples(results, kinds=("cascade", "baseline"))

## Conclusion

_Written after reading the results above._